# R3l — Reorganization Potential
One Run All: recover twelve official pre-Proposal Step300 anchors, generate fixed candidates and original Trust Region scans, independent SEARCH/SELECT, freeze finalists, deduplicated100-step Cell continuation, REPORT attribution and separate target-aware finite Beam diagnostic. T4 x2, Internet, existing HF_TOKEN/GITHUB_TOKEN. Confirmation SEALED.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

SOURCE_SHA = "RUNTIME_SHA_PENDING"
ROOT = Path("/kaggle/working/r3l-code-" + SOURCE_SHA[:12])
OUTPUT = Path("/kaggle/working/r3l-output-" + SOURCE_SHA[:12])
ASSETS = Path("/kaggle/working/r3l-calibration-assets")


def run(*args, **kwargs):
    print("[R3l]", *map(str, args), flush=True)
    if str(args[0]) == "git":
        kwargs.setdefault("timeout", 45)
    return subprocess.run(list(map(str, args)), check=True, **kwargs)


# Use installed PyTorch; only add the existing storage dependency if absent.
if importlib.util.find_spec("yaml") is None:
    run(sys.executable, "-m", "pip", "install", "--quiet", "pyyaml")
if importlib.util.find_spec("huggingface_hub") is None:
    run(sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub>=0.23,<2")
for package, requirement in (
    ("datasets", "datasets>=2.18,<5"),
    ("tokenizers", "tokenizers>=0.15,<1"),
):
    if importlib.util.find_spec(package) is None:
        run(sys.executable, "-m", "pip", "install", "--quiet", requirement)
ROOT.mkdir(parents=True, exist_ok=True)
run("git", "init", "--quiet", ROOT)
remote = subprocess.run(
    ["git", "-C", str(ROOT), "remote", "get-url", "origin"], capture_output=True, text=True
)
if remote.returncode:
    run("git", "-C", ROOT, "remote", "add", "origin", "https://github.com/ArcheLabs/mini-cells.git")
run("git", "-C", ROOT, "config", "remote.origin.promisor", "true")
run("git", "-C", ROOT, "config", "remote.origin.partialclonefilter", "blob:none")
run("git", "-C", ROOT, "sparse-checkout", "init", "--no-cone")
run(
    "git",
    "-C",
    ROOT,
    "sparse-checkout",
    "set",
    "--no-cone",
    *[
        "/scripts/run_r3l.py",
        "/pyproject.toml",
        "/src/",
        "/shared/python/",
        "/research/projects/cellular-language-models/engineering/kaggle-continuation/",
        "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/",
        "/research/projects/cellular-language-models/experiments/006-path-first-clm/",
        "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/",
        "/research/projects/continual-learning/context/04-continual-learning/scripts/research/",
        "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/",
    ],
)
head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
if head.returncode or head.stdout.strip() != SOURCE_SHA:
    run("git", "-C", ROOT, "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA)
    run("git", "-C", ROOT, "checkout", "--detach", "FETCH_HEAD")
actual = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError("Source identity mismatch")
env = dict(
    os.environ,
    HF_HUB_DISABLE_PROGRESS_BARS="1",
    HF_HUB_ETAG_TIMEOUT="10",
    HF_HUB_DOWNLOAD_TIMEOUT="30",
    TOKENIZERS_PARALLELISM="false",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
)
print(
    {
        "experiment": "CLM-001-R3l-Connection-Reorganization-Potential-001",
        "source": SOURCE_SHA,
        "anchors": 12,
        "new_cell_updates_upper_bound": 4800,
        "confirmation": "SEALED",
    },
    flush=True,
)
try:
    run(
        sys.executable,
        ROOT / "scripts/run_r3l.py",
        "--config",
        ROOT
        / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/config/frozen.json",
        "--run-all",
        "--output",
        OUTPUT,
        "--assets",
        ASSETS,
        cwd=ROOT,
        env=env,
    )
finally:
    print("Save Version Output for R3l reports:", OUTPUT, flush=True)